# 00 - Colab Runner (data pipeline + demand model)

Runs the heavy compute on Colab: raw data download, preprocessing, and demand model training.

Workflow: the repo is cloned fresh from GitHub into Colab's own storage every session, so everyone
runs the same code and no Drive mount is needed. Raw and processed data are rebuilt per session
(fast on Colab's connection and local disk). The trained artifact is small and gets shared back
through git (results/demand: mlp.pt, meta.json, metrics.json).

Before running:
1. Runtime -> Change runtime type -> GPU.
2. If the repo is private: add a fine grained GitHub token as a Colab secret named GITHUB_TOKEN
   (key icon in the left sidebar) with repo read and write access.
3. Runtime -> Run all.

In [ ]:
import os

GITHUB_REPO = 'LazarSazdov/marl-dqn-dynamic-pricing'
REPO_DIR = '/content/marl-dqn-dynamic-pricing'

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
clone_url = (f'https://{token}@github.com/{GITHUB_REPO}.git' if token
             else f'https://github.com/{GITHUB_REPO}.git')

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {clone_url} {REPO_DIR}
    %cd {REPO_DIR}

%pip install -q -r requirements.txt
%pip install -q -e .

import torch
print('CUDA available:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU only)')

## 1. Raw data (about 260MB, straight onto the fast local disk)

In [ ]:
!python3 scripts/download_data.py

## 2. Preprocess (about 5 minutes)

In [ ]:
!python3 scripts/preprocess.py
!ls -lh data/processed/

## 3. Train the demand models (GPU)

In [ ]:
!python3 scripts/train_demand.py

## 4. Summary

In [ ]:
import json

metrics = json.loads(open('results/demand/metrics.json').read())
print('=== TEST metrics ===')
for name, m in metrics['test'].items():
    print(f"  {name:15s} log-loss {m['log_loss']:.4f}  PR-AUC {m['pr_auc']:.4f}  "
          f"Brier {m['brier']:.4f}  ECE {m['ece']:.4f}")
print('\nelasticity correction:', json.dumps(metrics['elasticity_correction']))
print('\n=== Gates ===')
print(json.dumps(metrics['gates'], indent=2))
print('\nLR price_ratio coefficient:', metrics['lr_standardized_coefficients']['price_ratio'])

## 5. Share the trained artifact through git

The artifact is tiny (a few hundred KB) and gitignore already whitelists exactly these three
files. Fill in your own name and email, then run. Skip this cell if the gates failed.

In [ ]:
GIT_NAME = ''   # your name, for example 'Lazar Sazdov'
GIT_EMAIL = ''  # your github email

if GIT_NAME and GIT_EMAIL:
    !git config user.name "{GIT_NAME}"
    !git config user.email "{GIT_EMAIL}"
    !git add results/demand/mlp.pt results/demand/meta.json results/demand/metrics.json
    !git commit -m "feat: add trained demand model artifact from colab run"
    !git push
else:
    print('set GIT_NAME and GIT_EMAIL above to push the artifact')

## Done

If all three gates pass and the artifact is pushed, everyone can pull the same world model.
Colab storage is wiped when the runtime disconnects; that is fine, every input here is rebuilt
by script on the next run.